In [29]:
import scanpy as sc
import pandas as pd
import numpy as np
import harmonypy as hm
from sklearn.metrics import (homogeneity_score,v_measure_score,adjusted_mutual_info_score,normalized_mutual_info_score,adjusted_rand_score,fowlkes_mallows_score)


In [30]:
repeat_times = 2
simple_path = '/home/cavin/jt/benchmark/data/hbc/s1_filtered_cells.h5ad'
celltype_col = 'cell_type'
batch_key = "batch"
cell_emb_col = 'X_harmony'
random_seed=2026 + repeat_times * 200
save_path = "/home/cavin/jt/benchmark/experiments/embedings/batch_integrate/Harmony_human_breast_cancer_emb.parquet"


In [31]:
loaded_emb_df = pd.read_parquet(save_path)
adata = sc.read_h5ad(simple_path)
aligned_emb_df = loaded_emb_df.reindex(adata.obs_names)
adata

AnnData object with n_obs × n_vars = 281780 × 313
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'batch', 'cell_type', 'n_genes'
    var: 'ensemble_id', 'type'
    obsm: 'spatial'

In [32]:
adata.obsm[cell_emb_col] = aligned_emb_df.to_numpy(dtype=np.float32)
adata

AnnData object with n_obs × n_vars = 281780 × 313
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'batch', 'cell_type', 'n_genes'
    var: 'ensemble_id', 'type'
    obsm: 'spatial', 'X_harmony'

In [33]:
res_list = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]
key_added='leiden'
sc.pp.neighbors(adata, use_rep=cell_emb_col,random_state=random_seed)
adata


/home/cavin/anaconda3/envs/tao/lib/python3.9/site-packages/scipy/sparse/_index.py:143: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil_matrix is more efficient.
  self._set_arrayXarray(i, j, x)


AnnData object with n_obs × n_vars = 281780 × 313
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'batch', 'cell_type', 'n_genes'
    var: 'ensemble_id', 'type'
    uns: 'neighbors'
    obsm: 'spatial', 'X_harmony'
    obsp: 'distances', 'connectivities'

In [34]:
max_value = 0
metrics = {"method": "Harmony", "ARI": 0, "NMI": 0, "AMI": 0, "HS": 0, "VM": 0, "FMI": 0, "mean value": 0}
save_label_df = None

In [35]:

for used_res in res_list:
    sc.tl.leiden(adata, random_state=random_seed, resolution=used_res,key_added=key_added)
    true_labels = np.array(adata.obs[celltype_col])
    cluster_labels = np.array(adata.obs[key_added])
    FMI = fowlkes_mallows_score(true_labels, cluster_labels)
    homogeneity = homogeneity_score(true_labels, cluster_labels)
    v_measure = v_measure_score(true_labels, cluster_labels)
    ami = adjusted_mutual_info_score(true_labels, cluster_labels)
    nmi = normalized_mutual_info_score(true_labels, cluster_labels)
    ari = adjusted_rand_score(true_labels, cluster_labels)
    mean_value = np.mean([FMI, homogeneity, v_measure, ami, nmi, ari])

    n_cluster = len(adata.obs[key_added].unique())
    n_celltype = len(adata.obs[celltype_col].unique())
    if mean_value > max_value:
        save_label_df = adata.obs[key_added]
        metrics["ARI"] = ari
        metrics["NMI"] = nmi
        metrics["AMI"] = ami
        metrics["HS"] = homogeneity
        metrics["VM"] = v_measure
        metrics["FMI"] = FMI
        metrics["mean value"] = mean_value
        max_value = mean_value

    print(f'resolution = {used_res} | n_celltype = {n_celltype} | n_cluster = {n_cluster} | mean_value = {round(mean_value,3)} | ARI = {round(ari,3)} |  NMI = {round(nmi,3)} | AMI = {round(ami,3)} | HS = {round(homogeneity,3)} | VM = {round(v_measure,3)} | FMI = {round(FMI,3)} ')

resolution = 0.1 | n_celltype = 20 | n_cluster = 2 | mean_value = 0.366 | ARI = 0.25 |  NMI = 0.396 | AMI = 0.396 | HS = 0.256 | VM = 0.396 | FMI = 0.504 
resolution = 0.2 | n_celltype = 20 | n_cluster = 8 | mean_value = 0.518 | ARI = 0.386 |  NMI = 0.575 | AMI = 0.575 | HS = 0.494 | VM = 0.575 | FMI = 0.504 
resolution = 0.3 | n_celltype = 20 | n_cluster = 11 | mean_value = 0.547 | ARI = 0.426 |  NMI = 0.596 | AMI = 0.596 | HS = 0.563 | VM = 0.596 | FMI = 0.507 
resolution = 0.4 | n_celltype = 20 | n_cluster = 13 | mean_value = 0.583 | ARI = 0.483 |  NMI = 0.62 | AMI = 0.62 | HS = 0.605 | VM = 0.62 | FMI = 0.552 
resolution = 0.5 | n_celltype = 20 | n_cluster = 13 | mean_value = 0.588 | ARI = 0.488 |  NMI = 0.624 | AMI = 0.624 | HS = 0.611 | VM = 0.624 | FMI = 0.556 
resolution = 0.6 | n_celltype = 20 | n_cluster = 14 | mean_value = 0.579 | ARI = 0.458 |  NMI = 0.623 | AMI = 0.623 | HS = 0.618 | VM = 0.623 | FMI = 0.529 
resolution = 0.7 | n_celltype = 20 | n_cluster = 16 | mean_value

In [36]:
metrics

{'method': 'Harmony',
 'ARI': 0.48818299916566643,
 'NMI': 0.6237006905432138,
 'AMI': 0.6236334735689729,
 'HS': 0.6110651289214635,
 'VM': 0.6237006905432138,
 'FMI': 0.555710913359002,
 'mean value': 0.5876656493502553}

In [37]:
save_label_df

s1r1_1         6
s1r1_2         6
s1r1_5         3
s1r1_8         6
s1r1_9         5
              ..
s1r2_118748    7
s1r2_118749    7
s1r2_118750    7
s1r2_118751    7
s1r2_118752    7
Name: leiden, Length: 281780, dtype: category
Categories (13, object): ['0', '1', '2', '3', ..., '9', '10', '11', '12']

In [38]:
save_label_df_path = f"/home/cavin/jt/benchmark/experiments/results/labels_df/breast_cancer/harmony_human_breast_cancer_labels_repeat_{repeat_times}.csv"

In [39]:
save_label_df.to_csv(save_label_df_path)

In [40]:
metrics_df = pd.DataFrame.from_dict(metrics, orient='index').T

In [41]:
metrics_df_save_path = f"/home/cavin/jt/benchmark/experiments/results/cluster_metrics/breast_cancer/human_breast_cancer_metrics_repeat_{repeat_times}.csv"

In [42]:
metrics_df.to_csv(metrics_df_save_path, index=False,mode="a", header=not pd.io.common.file_exists(metrics_df_save_path))